In [15]:
# Cell 1 — Setup & imports
import polars as pl
from Python import statcast, pitcher_features

years = [2025]

In [16]:
# Cell 2 — Inspect vocabulary/constants
print(pitcher_features.PITCH_TYPES)
print(pitcher_features.CANON_PITCH)
print(pitcher_features.FANGRAPHS_FIP_CONSTANT)
print(pitcher_features.BUILD_COLUMNS)

('ff', 'si', 'fc', 'sl', 'st', 'cu', 'ch', 'fs')
{'FF': 'ff', 'FA': 'ff', 'SI': 'si', 'FT': 'si', 'FC': 'fc', 'SL': 'sl', 'SV': 'sl', 'ST': 'st', 'CU': 'cu', 'KC': 'cu', 'CS': 'cu', 'CH': 'ch', 'FS': 'fs', 'SF': 'fs'}
{2021: 3.17, 2022: 3.112, 2023: 3.255, 2024: 3.166, 2025: 3.135, 2026: 3.099}
('game_pk', 'game_date', 'player_name', 'pitcher', 'stand', 'p_throws', 'home_team', 'away_team', 'inning', 'inning_topbot', 'at_bat_number', 'pitch_number', 'pitch_type', 'type', 'description', 'events', 'bb_type', 'zone', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z', 'release_extension', 'release_pos_x', 'release_pos_z', 'vy0', 'vz0', 'ay', 'az', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'bat_score', 'post_bat_score')


In [17]:
# Cell 3 — Load raw data with the columns pitcher_features actually needs
# (statcast.DEFAULT_COLUMNS is missing vy0/vz0/ay/az/bb_type/zone/etc.)
raw = statcast.load_statcast_years(years, columns=list(pitcher_features.BUILD_COLUMNS))
raw.shape, raw.columns

((706571, 35),
 ['game_pk',
  'game_date',
  'player_name',
  'pitcher',
  'stand',
  'p_throws',
  'home_team',
  'away_team',
  'inning',
  'inning_topbot',
  'at_bat_number',
  'pitch_number',
  'pitch_type',
  'type',
  'description',
  'events',
  'bb_type',
  'zone',
  'release_speed',
  'release_spin_rate',
  'pfx_x',
  'pfx_z',
  'release_extension',
  'release_pos_x',
  'release_pos_z',
  'vy0',
  'vz0',
  'ay',
  'az',
  'estimated_ba_using_speedangle',
  'estimated_woba_using_speedangle',
  'woba_value',
  'woba_denom',
  'bat_score',
  'post_bat_score'])

In [18]:
# Cell 4 — Pitch-level enrichment (VAA, canon_pitch, outs_on_play, run_delta)
pl_df = pitcher_features._pitch_level(raw)
pl_df.select(["pitch_type", "canon_pitch", "vaa", "ivb", "hb", "outs_on_play", "run_delta"]).head(10)

# Sanity check VAA range: should generally fall between -4 and -12 degrees for MLB pitches
pl_df.select(
    pl.col("vaa").min().alias("vaa_min"),
    pl.col("vaa").max().alias("vaa_max"),
    pl.col("vaa").mean().alias("vaa_mean"),
)

pl_df.group_by("canon_pitch").agg(
    pl.len().alias("n_pitches"),
    pl.col("vaa").mean().alias("vaa_mean"),
    pl.col("vaa").min().alias("vaa_min"),
    pl.col("vaa").max().alias("vaa_max"),
    pl.col("vaa").std().alias("vaa_std"),
).sort("vaa_mean")

canon_pitch,n_pitches,vaa_mean,vaa_min,vaa_max,vaa_std
str,u32,f64,f64,f64,f64
null,4609,-9.808654,-31.147028,-1.941794,3.550472
"""cu""",56856,-9.698809,-25.222145,-2.489515,1.392124
"""sl""",110284,-7.722156,-15.752392,0.500766,1.277126
"""fs""",21946,-7.679838,-13.417853,-1.955502,1.152626
"""st""",47575,-7.678181,-14.198411,-2.486464,1.258325
"""ch""",71844,-7.437712,-13.42691,3.327281,1.111291
"""fc""",57523,-6.255681,-12.598327,0.151875,1.220197
"""si""",110569,-5.897322,-11.243397,-1.156729,0.980401
"""ff""",225365,-4.731541,-22.128259,-0.241921,1.030646


In [19]:
# Cell 5 — Starter identification
starters = pitcher_features._starter_keys(pl_df)
starters.shape

# Confirm exactly one starter per (game_pk, inning_topbot) -- catches double-header
# or data anomalies silently producing wrong starter attribution
check = (
    pl_df.filter(pl.col("inning") == 1)
    .group_by(["game_pk", "inning_topbot"])
    .agg(pl.col("pitcher").n_unique().alias("n_pitchers_seen"))
)
starters.shape[0], check.shape[0]  # starters rows should equal number of (game_pk, inning_topbot) pairs

(4822, 4822)

In [20]:
# Cell 6 — Full aggregation: one row per starting-pitcher-game
starts = pitcher_features.build_pitcher_starts(raw, min_batters_faced=9)
starts.shape

# Uniqueness check -- should be 0 duplicates on the primary key
starts.select(["game_pk", "pitcher"]).is_duplicated().sum()

0

In [21]:
# Cell 7 — Inspect core label columns (K, PA, Outs) -- these are LABELS, not features
starts.select(["player_name", "game_date", "PA", "K", "BB", "HBP", "HR", "Outs", "wOBA", "xWOBA" if "xWOBA" in starts.columns else "xwOBA"]).head(10)

# Distribution sanity check on the eventual model target
starts_check = starts.with_columns((pl.col("K") / pl.col("PA")).alias("k_rate"))
starts_check.select(
    pl.col("k_rate").min().alias("k_rate_min"),
    pl.col("k_rate").max().alias("k_rate_max"),
    pl.col("k_rate").mean().alias("k_rate_mean"),
)

k_rate_min,k_rate_max,k_rate_mean
f64,f64,f64
0.0,0.684211,0.219213


In [22]:
# Cell 8 — Spot-check one known start end-to-end
starts.filter(pl.col("player_name").str.contains("Cristopher")).select(
    ["game_date", "PA", "K", "Outs", "wOBA", "xwOBA", "extension", "rel_x", "rel_z"]
)
# Replace "Skubal" with any pitcher name you want to manually verify against a box score

game_date,PA,K,Outs,wOBA,xwOBA,extension,rel_x,rel_z
date,u32,u32,i32,f64,f64,f64,f64,f64
2025-04-02,20,8,15,0.3125,0.207457,6.743529,1.935059,6.433882
2025-04-08,22,1,13,0.356818,0.4018815,6.796552,2.023448,6.330345
2025-04-13,26,6,18,0.288462,0.230823,6.979121,1.875055,6.430659
2025-04-18,24,10,18,0.254167,0.187005,6.782474,1.907835,6.41433
2025-04-24,18,3,9,0.436111,0.298022,6.816216,1.91027,6.345811
…,…,…,…,…,…,…,…,…
2025-09-05,27,7,21,0.264815,0.235889,7.176042,1.828333,5.869063
2025-09-10,23,4,18,0.186957,0.189267,7.011702,1.905745,6.098511
2025-09-16,27,7,20,0.266667,0.33819,7.063218,1.932644,6.101954


In [23]:
# Cell 9 — Arsenal / pitch-type usage check
usage_cols = [f"{pt}_usage_vR" for pt in pitcher_features.PITCH_TYPES] + \
             [f"{pt}_usage_vL" for pt in pitcher_features.PITCH_TYPES]
starts.select(["player_name"] + usage_cols).head(5)

# Sanity check: usage rates vs each hand should sum to <= 1.0 (not necessarily exactly 1.0
# since unmapped pitch types like knuckleballs/eephus aren't in PITCH_TYPES)
vR_cols = [f"{pt}_usage_vR" for pt in pitcher_features.PITCH_TYPES]
starts.with_columns(pl.sum_horizontal(vR_cols).alias("total_usage_vR")).select(
    pl.col("total_usage_vR").min().alias("usage_vR_min"),
    pl.col("total_usage_vR").max().alias("usage_vR_max"),
)

usage_vR_min,usage_vR_max
f64,f64
0.0,1.0


In [24]:
# Cell 10 — Confirm no leaked helper columns survived
helper_prefixes = ("_pit_", "_ff_", "_si_", "_fc_", "_sl_", "_st_", "_cu_", "_ch_", "_topbot")
leaked = [c for c in starts.columns if c.startswith(helper_prefixes)]
leaked  # should be an empty list

[]

In [25]:
# Cell 11 — League HR/FB (must use ALL pitches, not just starters, per dev-notes)
hr_fb = pitcher_features.league_hr_fb_from_pitches(raw)
hr_fb

{2025: 0.131230802718456}

In [26]:
# Cell 12 — FIP / xFIP enrichment
enriched = pitcher_features.add_fip_xfip(starts, league_hr_fb=hr_fb)
enriched.select(["player_name", "season", "FIP", "xFIP"]).sort("FIP").head(10)

# Sanity check: league-average FIP should land near the season's published cFIP constant
enriched.group_by("season").agg(pl.col("FIP").mean().alias("avg_FIP"), pl.col("xFIP").mean().alias("avg_xFIP"))

season,avg_FIP,avg_xFIP
i32,f64,f64
2025,4.270242,4.136095


In [27]:
# Cell 13 — Verify FIP core-only mode (include_constant=False) shifts every value by the constant
core_only = pitcher_features.add_fip_xfip(starts, league_hr_fb=hr_fb, include_constant=False)
diff_check = enriched.select("player_name", "game_pk", "season", "FIP").join(
    core_only.select("player_name", "game_pk", "season", "FIP"),
    on=["player_name", "game_pk", "season"],
    suffix="_core",
).with_columns((pl.col("FIP") - pl.col("FIP_core")).alias("diff"))

diff_check.select(
    pl.col("diff").min().alias("diff_min"),
    pl.col("diff").max().alias("diff_max"),
)

diff_min,diff_max
f64,f64
3.135,3.135
